# Seleção das Imagens com Torres dos Datasets
***
Datasets utilizados para montagem do diretório: ```TTPLA Dataset``` e ```STN-PLAD Dataset```.

### Importação dos Pacotes Utilitários
***

In [22]:
from collections import Counter
import json, os, shutil

In [27]:
def copy_archives(source_path, images_files, destination_path):
    '''
    Copia arquivos de uma pasta para outra.

    Args:
        source_path (str) --> Caminho relativo da pasta fonte onde estão os arquivos.
        images_files (list) --> Lista com o nome das imagens contidas no diretório fonte.
        destination_path (str) --> Caminho relativo da paste onde os arquivos serão destinados.
    '''

    try:
        if not os.path.exists(destination_path):
            os.makedirs(destination_path)
    
        for image in images_files:
            source_ref = os.path.join(source_path, image)
            destination_ref = os.path.join(destination_path, image)
            shutil.copy(source_ref, destination_ref)
        
        print("Arquivos copiados com sucesso!")
    except Exception as e:
        print("Ocorreu um erro:", str(e))

### Seleção das Imagens com Torres: ```TTPLA Dataset```
***

In [2]:
def list_json_files(folder_path):
    '''
    Separa uma lista com o nome dos arquivos .json localizados em um diretório.
    
    Args:
        folder_path (str) --> caminho relativo do diretório a ser analisado.
    
    Returns:
        json_files (list) --> lista contendo o nome de todos os arquivos .json selecionados.
    
    '''

    json_files = []

    for file in os.listdir(folder_path):
        if file.endswith('.json'):
            json_files.append(file)
            
    return json_files

In [3]:
def extract_number(filename):
    '''
    Função utilitária para organizar a ordem alfabética dos arquivos .json salvos em uma lista.

    Args:
        filename (list) --> nome dos arquivos .json salvos na string.

    Returns:
        filename_sorted (list) --> string com o nome dos arquivos .json em ordem alfabética.
    '''

    start = filename.find('_') + 1
    end = filename.find('_', start)

    if end == -1: end = filename.find('.', start)
        
    return int(filename[start:end])

In [4]:
# carregando os arquivos .json em uma lista
labels_folder = '../TTPLA-dataset/'
jsons_files = list_json_files(labels_folder)
jsons_files = sorted(jsons_files, key = extract_number)

In [14]:
def count_categories(json_path):
    '''
    Conta a quantidade de objetos instanciados em um arquivo .json.

    Args:
        json_path (list) --> lista contendo o caminho relativo para acessar os arquivos .json.
    
    Returns:
        category_counts (dict) --> dicionário contendo a contagem de objetos instanciados na
                                   imagem.
    '''
    
    with open(json_path, 'r') as json_file:
        data = json.load(json_file)

    category_counts = dict()

    for shape in data['shapes']:
        category = shape['label']
        if category in category_counts:
            category_counts[category] += 1
        else:
            category_counts[category] = 1

    category_counts['imagePath'] = data['imagePath']

    return category_counts

In [23]:
# contando a quantidade de objetos instanciados por classe em cada uma das imagens
count_category_counts = list()

for json_file in jsons_files:
    category_counts = count_categories('../TTPLA-dataset/' + json_file)
    count_category_counts.append(category_counts)

In [28]:
# cria uma lista com o nome das imagens que possuem torres de transmissão
images_with_towers = list()

for category in count_category_counts:
    keys = category.keys()
    for key in keys:
        count = category[key]
        if (key == 'tower_tucohy') | (key == 'tower_lattice'): 
            images_with_towers.append(category['imagePath'])
        break
        
print(f'{len(images_with_towers)} Imagens que contém torres de transmissão')

279 Imagens que contém torres de transmissão


In [36]:
# copiando para uma pasta em separado as imagens que apenas contém torres
source_path = '../TTPLA-dataset/'
destination_path = 'TTPLA-dataset/'
copy_archives(source_path, images_with_towers, destination_path)

Arquivos copiados com sucesso!


In [38]:
# pegando a lista de jsons das imagens selecionadas
jsons_with_towers = [images.replace(".jpg", ".json") for images in images_with_towers]

In [41]:
# copiando para uma pasta em separado os arquivos jsons das imagens que contém torres
copy_archives(source_path, jsons_with_towers, destination_path)

Arquivos copiados com sucesso!


### Seleção das Imagens com Torres: ```STN-PLAD Dataset```

In [2]:
def load_json_file(json_path):
    '''
    Carrega um arquivo json em um dicionário.

    Args:
        json_path (str) --> caminho relativo de acesso ao arquivo json no diretório.

    Returns:
        data (dict) --> dicionário contendo todas as informações do arquivo json.
    '''

    with open(json_path, 'r') as json_file:
        data = json.load(json_file)
    
    return data

In [3]:
def initialize_images_ids_with_dict(data):
    '''
    Adquirindo os IDs das imagens a partir do dicionário com os rótulos do STN-PLAD.

    Args:
        data (dict) --> dicionário com as informações de rotulagem do conjunto de imagem.
    
    Returns:
        count_ids (dict) --> dicionário vazio com as chaves de IDs das imagens.
    '''

    count_ids = dict()
    for info in data['images']:
        count_ids[info['id']] = list()
    
    return count_ids

In [16]:
def image_path_search(id_verify, data):
    '''
    Pesquisa na lista de dicionários o caminho relativo da imagem a partir do ID.

    Args:
        id_verify (int) --> id relativo para pesquisar a imagem na lista de dicionário.
        data (list) --> lista de dicionários para pesquisa.
    
    Returns:
        image_path (str) --> caminho relativo da imagem.
    '''

    for image_data in data['images']:
        if image_data['id'] == id_verify:
            return image_data['file_name']

In [5]:
# carregando os rótulos no arquivo json dentro de um dicionário
json_path = '../STN-PLAD-dataset/annotations.json'
data = load_json_file(json_path)

In [17]:
# inicializando um dicionário vazio com os IDs das imagens
images_ids = initialize_images_ids_with_dict(data)

# obtendo a quantidade de instâncias de objeto por imagem
for info in data['annotations']:
    images_ids[info['image_id']].append(info['category_id'])

In [25]:
# carrega em uma lista o caminho relativo das imagens com torres
images_with_towers = list()

for index, images_instances in enumerate(images_ids.values()):
    count_aux = Counter(images_instances)
    if count_aux[0] != 0:
        images_with_towers.append(image_path_search(index, data)) 

print(f'Quantidade de imagens com torres: {len(images_with_towers)} imagens')

Quantidade de imagens com torres: 131 imagens


In [29]:
# copiando para uma pasta em separado as imagens que apenas contém torres
source_path = '../STN-PLAD-dataset/'
destination_path = 'STN-PLAD-dataset/'
copy_archives(source_path, images_with_towers, destination_path)

Arquivos copiados com sucesso!
